# Alexander gene sets — Enrichr enrichment

In [1]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import requests
from IPython.display import display
from maayanlab_bioinformatics.api.enrichr import enrichr_link_from_genes, enrichr_get_top_results

GENE_SETS = Path("gene_sets")
MUTATIONS = ["R239H", "R239C", "R79C", "R79H", "R239G"]
DIRECTIONS = ["up", "down"]
ENRICHR = "https://maayanlab.cloud/Enrichr"
LIBRARIES = [s["libraryName"] for s in requests.get(f"{ENRICHR}/datasetStatistics", timeout=60).json()["statistics"]]
TOP_N = 10
len(LIBRARIES)

228

In [2]:
def load(mut, direction):
    p = GENE_SETS / f"{mut}_{direction}.txt"
    return [l.strip() for l in p.read_text().splitlines() if l.strip()]

gene_sets = {(m, d): load(m, d) for m in MUTATIONS for d in DIRECTIONS}
pd.Series({f"{m}_{d}": len(g) for (m, d), g in gene_sets.items()}, name="n_genes")

R239H_up       60
R239H_down    121
R239C_up       33
R239C_down     67
R79C_up        63
R79C_down      68
R79H_up        57
R79H_down      74
R239G_up       54
R239G_down     89
Name: n_genes, dtype: int64

In [3]:
ENR_COLS = ["rank", "term", "pvalue", "zscore", "combinedscore", "overlapping_genes", "adjusted_pvalue"]
frames, links = [], {}
for (m, d), genes in gene_sets.items():
    if not genes:
        print(f"{m}_{d}: empty gene set, skipped")
        continue
    res = enrichr_link_from_genes(genes, f"{m}_{d}", sleep=0)
    uid = res["userListId"]
    links[f"{m}_{d}"] = res["link"]
    def fetch(lib, uid=uid, m=m, d=d):
        try:
            df = enrichr_get_top_results(uid, lib, sleep=0)
        except Exception:
            return None
        if df is None or len(df) == 0:
            return None
        df = df[ENR_COLS].nsmallest(TOP_N, "adjusted_pvalue").copy()
        df.insert(0, "library", lib)
        df.insert(0, "direction", d)
        df.insert(0, "mutation", m)
        return df
    with ThreadPoolExecutor(max_workers=8) as ex:
        for df in ex.map(fetch, LIBRARIES):
            if df is not None:
                frames.append(df)
    print(f"{m}_{d}: done ({len(LIBRARIES)} libraries)")

enr = pd.concat(frames, ignore_index=True)
enr["n_overlap"] = enr["overlapping_genes"].str.len()
enr.shape

R239H_up: done (228 libraries)


R239H_down: done (228 libraries)


R239C_up: done (228 libraries)


R239C_down: done (228 libraries)


R79C_up: done (228 libraries)


...

R79H_up: done (228 libraries)


R79H_down: done (228 libraries)


R239G_up: done (228 libraries)


R239G_down: done (228 libraries)


(22226, 11)

## Shareable Enrichr links (one per gene set)

In [4]:
from IPython.display import HTML
HTML("<br>".join(f'{k}: <a href="{u}" target="_blank">{u}</a>' for k, u in links.items()))

In [5]:
def top_terms(mut, direction, n=TOP_N):
    sub = enr[(enr.mutation == mut) & (enr.direction == direction)]
    cols = ["library", "term", "pvalue", "adjusted_pvalue", "combinedscore", "n_overlap", "overlapping_genes"]
    return sub.sort_values("adjusted_pvalue").head(n)[cols].reset_index(drop=True)

pd.set_option("display.max_colwidth", 70)

## Top 10 enriched terms per gene set (across all Enrichr libraries)

In [6]:
for d in DIRECTIONS:
    print(f"\n========== R239H {d} ==========")
    display(top_terms("R239H", d))


========== R239H up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,ProteomicsDB_2020,Lung non-small lung cancer BTO:0002553 P005587-I,7.323440e-07,0.000660,178.398849,8,"[GSTM4, C3, GSTM1, ITIH2, APOA1, HMOX1, ANTXR1, ZBTB2]"
1,Genes_Associated_with_NIH_Grants,R00ES015774,8.906703e-07,0.001527,3654.397738,3,"[GSTM1, HFE, HMOX1]"
2,GO_Cellular_Component_2021,endoplasmic reticulum lumen (GO:0005788),2.244468e-05,0.001998,99.993024,7,"[C3, ITIH2, ARSL, WNT5A, GPX7, APOA1, ARSD]"
3,TargetScan_microRNA_2017,mmu-miR-18a,6.457865e-06,0.002089,51.317482,17,"[EPHA5, NCOA1, DOCK4, URM1, AMIGO2, ADCY3, NAV1, PPP2R3A, ANTXR1, ..."
4,TargetScan_microRNA_2017,mmu-miR-18b,6.457865e-06,0.002089,51.317482,17,"[EPHA5, NCOA1, DOCK4, URM1, AMIGO2, ADCY3, NAV1, PPP2R3A, ANTXR1, ..."
5,GO_Cellular_Component_2023,Endoplasmic Reticulum Lumen (GO:0005788),2.194437e-05,0.002129,100.570465,7,"[C3, ITIH2, ARSL, WNT5A, GPX7, APOA1, ARSD]"
6,KEGG_2016,Glutathione metabolism Homo sapiens hsa00480,1.778773e-05,0.002312,323.748277,4,"[GSTM4, GSTM1, ODC1, GPX7]"
7,GO_Cellular_Component_2025,Endoplasmic Reticulum Lumen (GO:0005788),2.454116e-05,0.002380,97.732474,7,"[C3, ITIH2, ARSL, WNT5A, GPX7, APOA1, ARSD]"
8,Metabolomics_Workbench_Metabolites_2022,Glutathione,1.707922e-04,0.002733,275.429783,3,"[GSTM4, GSTM1, GPX7]"
9,KEGG_2019_Human,Glutathione metabolism,2.391753e-05,0.003492,290.695324,4,"[GSTM4, GSTM1, ODC1, GPX7]"



========== R239H down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,Rummagene_kinases,PMC5302127-NIHMS832530-supplement-supp table4.xlsx-CSNK1A1-ANKRD52...,7.607076e-14,1.328196e-10,194.381937,31,"[COL17A1, ERRFI1, CEMIP, TINAGL1, PRKAG2, SLC2A3, HDAC9, ZFP36L1, ..."
1,Rummagene_kinases,PMC3937590-gb-2013-14-10-r110-S7.xlsx-AKT1-2 inhibitor-AKT1-2 Inhi...,1.823595e-13,1.591998e-10,167.266619,34,"[ERRFI1, NNMT, TINAGL1, LRP1, ZDBF2, CALD1, PRKAR2B, PEG10, STC2, ..."
2,Enrichr_Submissions_TF-Gene_Coocurrence,HMGA2,3.225594e-12,1.369264e-09,300.569101,17,"[ERRFI1, TGFB2, TINAGL1, IGFBP5, PLAUR, SLC2A3, NRG1, DKK1, SERPIN..."
3,Enrichr_Submissions_TF-Gene_Coocurrence,SP6,3.225594e-12,1.369264e-09,300.569101,17,"[ERRFI1, TGFB2, TINAGL1, IGFBP5, PLAUR, DKK1, SERPINB8, ASS1, BCAM..."
4,Enrichr_Submissions_TF-Gene_Coocurrence,PPP1R13L,3.225594e-12,1.369264e-09,300.569101,17,"[ERRFI1, TGFB2, TINAGL1, LRP1, PLAUR, SERPINB8, ASS1, ZFP36L1, GPR..."
5,Enrichr_Submissions_TF-Gene_Coocurrence,HOXB9,3.225594e-12,1.369264e-09,300.569101,17,"[ERRFI1, TGFB2, TINAGL1, IGFBP5, PLAUR, ASNS, SLC2A3, DKK1, SERPIN..."
6,Rummagene_signatures,PMC9545041-PCMR-35-517-s007.xlsx-Up Down genes-Down Regulated N 16...,1.898129e-13,1.396644e-09,155.400287,37,"[COL17A1, ERRFI1, CEMIP, NNMT, TINAGL1, LRP1, CXCL14, ZFP36L1, FUT..."
7,RummaGEO_GenePerturbations_2025,GSE173274 STBD1 Overexpression 2 V 0 Human up,6.229799e-13,2.265155e-09,176.908043,29,"[COL17A1, ERRFI1, SLC27A1, TINAGL1, LRP1, ACY1, SLC2A3, ZFP36L1, A..."
8,Enrichr_Submissions_TF-Gene_Coocurrence,ZNF114,3.640221e-11,5.150913e-09,253.618511,16,"[ERRFI1, TGFB2, CEMIP, PLAUR, SLC2A3, DKK1, SERPINB8, HDAC9, BST2,..."
9,Enrichr_Submissions_TF-Gene_Coocurrence,BNC1,3.640221e-11,5.150913e-09,253.618511,16,"[ERRFI1, TGFB2, CEMIP, NNMT, TINAGL1, IGFBP5, PLAUR, NRG1, DKK1, C..."


In [7]:
for d in DIRECTIONS:
    print(f"\n========== R239C {d} ==========")
    display(top_terms("R239C", d))


========== R239C up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,BioPlex_2017,RAP1GDS1,1.782469e-05,0.002620,778.683904,3,"[RAB34, RAC2, DNAL1]"
1,LINCS_L1000_Chem_Pert_down,CPC014 HCC515 6H-e3380-10.0,5.381015e-07,0.003294,1102.330811,4,"[EPHA7, HFE, RAC2, DHTKD1]"
2,CCLE_Proteomics_2020,SF295 CENTRAL NERVOUS SYSTEM TenPx23,1.298158e-05,0.003531,101.200888,8,"[EPHA7, CCNH, NAV1, CALML3, PPP2R3A, SHROOM1, DNAL1, CEP44]"
3,Proteomics_Drug_Atlas_2023,AZD8055 Down,4.303438e-06,0.003800,303.738712,5,"[ODC1, PPP2R3A, SLC7A11, DNAL1, B4GALT5]"
4,Jensen_DISEASES_Experimental_2025,OSTEOARTHRITIS,2.660412e-05,0.004469,175.293085,5,"[POLR2M, H1-6, ZNF618, HFE, HMGCLL1]"
5,TargetScan_microRNA_2017,hsa-miR-3201,1.853067e-05,0.005754,62.307953,12,"[POLR2M, ATP7B, ZNF618, DAB2IP, FAM3A, PPP2R3A, SLC7A11, FBXO33, H..."
6,TargetScan_microRNA_2017,hsa-miR-4791,1.853067e-05,0.005754,62.307953,12,"[POLR2M, ATP7B, ZNF618, DAB2IP, FAM3A, PPP2R3A, SLC7A11, FBXO33, H..."
7,LINCS_L1000_Chem_Pert_down,CPC010 MCF7 24H-trichostatin a-10.0,2.148783e-06,0.006577,248.869436,6,"[CCNH, HFE, RAC2, CALML3, DUS1L, DHTKD1]"
8,TargetScan_microRNA_2017,mmu-miR-18a,5.338652e-05,0.008288,53.301057,11,"[NCOA1, EPHA7, DOCK4, ATP7B, ZNF618, NAV1, PPP2R3A, SLC7A11, FBXO3..."
9,TargetScan_microRNA_2017,mmu-miR-18b,5.338652e-05,0.008288,53.301057,11,"[NCOA1, EPHA7, DOCK4, ATP7B, ZNF618, NAV1, PPP2R3A, SLC7A11, FBXO3..."



========== R239C down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,Orphanet_Augmented_2021,Benign hereditary chorea ORPHA:1429,4.930188e-08,0.000033,409.905720,7,"[ARNT2, MPP2, EPB41L1, LRP4, COBL, ANK2, AIF1L]"
1,Orphanet_Augmented_2021,Tetraamelia-multiple malformations syndrome ORPHA:3301,4.930188e-08,0.000033,409.905720,7,"[ARNT2, GAP43, AADAT, EPB41L1, LRP4, ANK2, WASF3]"
2,Orphanet_Augmented_2021,Achondroplasia ORPHA:15,1.110377e-06,0.000086,281.616989,6,"[EPB41L1, ERBB2, NOS1AP, LRP4, COBL, AIF1L]"
3,Orphanet_Augmented_2021,Autosomal dominant adult-onset proximal spinal muscular atrophy OR...,1.110377e-06,0.000086,281.616989,6,"[AADAT, EPB41L1, NOS1AP, TRIM37, WASF3, AIF1L]"
4,Orphanet_Augmented_2021,Camptodactyly-tall stature-scoliosis-hearing loss syndrome ORPHA:8...,1.110377e-06,0.000086,281.616989,6,"[EPB41L1, ERBB2, NOS1AP, LRP4, COBL, AIF1L]"
5,Orphanet_Augmented_2021,Crouzon syndrome-acanthosis nigricans syndrome ORPHA:93262,1.110377e-06,0.000086,281.616989,6,"[EPB41L1, ERBB2, NOS1AP, LRP4, COBL, AIF1L]"
6,Orphanet_Augmented_2021,Muenke syndrome ORPHA:53271,1.110377e-06,0.000086,281.616989,6,"[EPB41L1, ERBB2, NOS1AP, LRP4, COBL, AIF1L]"
7,Orphanet_Augmented_2021,Spectrin-associated autosomal recessive cerebellar ataxia ORPHA:35...,1.110377e-06,0.000086,281.616989,6,"[EPB41L1, NOS1AP, LRP4, COBL, OGDHL, AIF1L]"
8,Orphanet_Augmented_2021,Spinocerebellar ataxia type 5 ORPHA:98766,1.110377e-06,0.000086,281.616989,6,"[EPB41L1, NOS1AP, LRP4, COBL, OGDHL, AIF1L]"
9,Orphanet_Augmented_2021,Severe achondroplasia-developmental delay-acanthosis nigricans syn...,1.110377e-06,0.000086,281.616989,6,"[EPB41L1, ERBB2, NOS1AP, LRP4, COBL, AIF1L]"


In [8]:
for d in DIRECTIONS:
    print(f"\n========== R79C {d} ==========")
    display(top_terms("R79C", d))


========== R79C up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,GTEx_Tissue_Expression_Down,GTEX-S7SF-0001-SM-3K2BE blood female 20-29 years,5.243998e-08,0.000105,74.251642,27,"[DOCK4, HFE, CALML3, CXCL14, ANTXR1, SLC8A1, SELENBP1, ATP7B, CREG..."
1,ProteomicsDB_2020,Myeloma RPMI-8226 BTO:0000726 RPMI8226 NCI60,2.445443e-07,0.000223,223.031097,8,"[STARD13, ACTR6, CREG1, PLXNA2, GPX7, CALML3, ASS1, ZBTB2]"
2,ProteomicsDB_2020,Breast T-47D BTO:0001248 P001386,1.301183e-06,0.000594,131.366881,9,"[PTP4A2, STARD13, CREG1, PLXNA2, FAM3A, EFHD1, KDELR2, DHTKD1, ZBTB2]"
3,ProteomicsDB_2020,Lung NCI-H460 BTO:0002207 P003814,2.413408e-06,0.000734,115.918680,9,"[STARD13, GSTM1, POLE2, CREG1, PLXNA2, KDELR2, ANTXR1, CD99, ZBTB2]"
4,HDSigDB_Human_2021,Cortex RNA M34 Module (Darkmagenta) PMID26900923,5.505499e-07,0.000878,83.442697,15,"[EPHA5, NCOA1, CEMIP, DOCK4, TMOD2, ADCY3, NAV1, ARHGAP26, DHTKD1,..."
5,ProteomicsDB_2020,Melanoma A-375 BTO:0002806 SDC Rep1,6.584734e-06,0.001503,177.588864,6,"[STARD13, PLXNA2, ANTXR1, CD99, B4GALT5, ZBTB2]"
6,TISSUES_Curated_2025,CENTRAL NERVOUS SYSTEM,3.262709e-05,0.001653,30.106174,43,"[CEMIP, DOCK4, CCNH, GRIK5, ANTXR1, GSPT2, TAF5L, FGFRL1, SLC8A1, ..."
7,TISSUES_Curated_2025,BRAIN,2.700300e-05,0.001653,31.006001,43,"[CEMIP, DOCK4, CCNH, GRIK5, ANTXR1, GSPT2, TAF5L, FGFRL1, SLC8A1, ..."
8,TISSUES_Curated_2025,HEAD,2.287044e-05,0.001653,32.825321,46,"[CEMIP, DOCK4, CCNH, GRIK5, CXCL14, ANTXR1, GSPT2, TAF5L, FGFRL1, ..."
9,TISSUES_Curated_2025,NERVOUS SYSTEM,4.591862e-05,0.001745,28.513748,43,"[CEMIP, DOCK4, CCNH, GRIK5, ANTXR1, GSPT2, TAF5L, FGFRL1, SLC8A1, ..."



========== R79C down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,MSigDB_Oncogenic_Signatures,MEL18 DN.V1 UP,4.507695e-11,5.003542e-09,620.837447,10,"[NRP1, PYGB, TGFB2, ECM1, NRP2, STC2, SPP1, ITGA5, LCP1, CPA4]"
1,Enrichr_Submissions_TF-Gene_Coocurrence,BMP2,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, NRP1, TGFB2, NRP2, GPRC5A, IGFBP5, STC2, SPP1, ITGA5, SE..."
2,Enrichr_Submissions_TF-Gene_Coocurrence,BNC1,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, NRP1, TGFB2, NRP2, GPRC5A, IGFBP5, STC2, AMIGO2, SPP1, S..."
3,Enrichr_Submissions_TF-Gene_Coocurrence,ETV4,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, TGFB2, NRP2, GPRC5A, STC2, SPP1, SLC2A3, ITGA5, MELTF, S..."
4,Enrichr_Submissions_TF-Gene_Coocurrence,GATA6,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, NRP1, TGFB2, GPRC5A, IGFBP5, STC2, AMIGO2, SPP1, SLC2A3,..."
5,Enrichr_Submissions_TF-Gene_Coocurrence,GRHL3,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, NRP2, GPRC5A, IGFBP5, STC2, SPP1, IFFO2, SLC2A3, MELTF, ..."
6,Enrichr_Submissions_TF-Gene_Coocurrence,HMGA2,3.451327e-10,4.192035e-08,319.567514,12,"[NRP1, TGFB2, NRP2, GPRC5A, IGFBP5, STC2, SPP1, SLC2A3, ITGA5, SER..."
7,Enrichr_Submissions_TF-Gene_Coocurrence,HOXB9,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, NRP1, TGFB2, GPRC5A, IGFBP5, STC2, SPP1, SLC2A3, MELTF, ..."
8,Enrichr_Submissions_TF-Gene_Coocurrence,MKRN3,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, NRP1, TGFB2, GAP43, IGFBP5, PEG10, SPP1, SLC2A3, MELTF, ..."
9,Enrichr_Submissions_TF-Gene_Coocurrence,SP6,3.451327e-10,4.192035e-08,319.567514,12,"[ALDH1A3, TGFB2, NRP2, GPRC5A, IGFBP5, STC2, SPP1, LCP1, MELTF, SE..."


In [9]:
for d in DIRECTIONS:
    print(f"\n========== R79H {d} ==========")
    display(top_terms("R79H", d))


========== R79H up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,Rummagene_signatures,PMC9710780-pone.0278155.s001.xlsx-Up-reg DEG-Oli Up,2.450167e-07,0.001279,677.362700,5,"[NCOA1, DOCK4, STARD13, TNKS, CCNH]"
1,KEGG_2019_Human,Glutathione metabolism,1.949554e-05,0.002885,313.097764,4,"[GSTM1, ODC1, GPX7, CHAC2]"
2,KEGG_2021_Human,Glutathione metabolism,2.092257e-05,0.003369,305.173984,4,"[GSTM1, ODC1, GPX7, CHAC2]"
3,KEGG_2019_Mouse,Glutathione metabolism,3.315920e-05,0.004443,257.958546,4,"[GSTM1, ODC1, GPX7, CHAC2]"
4,Azimuth_2023,Kidney-L1-Schwann,2.836202e-04,0.004538,845.894772,2,"[STARD13, ZNF536]"
5,SubCell_BarCode,HCC827 Cytosol Unclassified 30609389,1.292036e-04,0.007235,29.060186,18,"[STARD13, TNKS, ODC1, DAB2IP, NAV1, CALML3, SHROOM1, PPP2R3A, FBXO..."
6,TargetScan_microRNA_2017,mmu-miR-487b,1.647920e-05,0.010497,139.139384,6,"[DOCK4, CCNT2, KDELR2, FBXO33, CEP44, WASF3]"
7,Phosphatase_Substrates_from_DEPOD,PTPN3,1.135209e-02,0.011352,531.537849,1,[MAPK12]
8,TargetScan_microRNA_2017,mmu-miR-18a,5.796207e-05,0.012307,37.802283,15,"[NCOA1, STARD13, DOCK4, AMIGO2, NAV1, PPP2R3A, ANTXR1, FBXO33, MAP..."
9,TargetScan_microRNA_2017,mmu-miR-18b,5.796207e-05,0.012307,37.802283,15,"[NCOA1, STARD13, DOCK4, AMIGO2, NAV1, PPP2R3A, ANTXR1, FBXO33, MAP..."



========== R79H down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,Rummagene_kinases,PMC3937590-gb-2013-14-10-r110-S7.xlsx-AKT1-2 inhibitor-AKT1-2 Inhi...,3.521027e-08,0.000050,91.902092,20,"[TGFB2, TINAGL1, LRP1, IGFBP5, CD70, MTARC1, DKK1, PRSS12, EFR3B, ..."
1,Enrichr_Submissions_TF-Gene_Coocurrence,TFCP2L1,1.481598e-07,0.000120,166.950324,10,"[SELENBP1, TGFB2, TINAGL1, KLF5, PARM1, IGFBP5, FANCA, SLC2A3, LCP..."
2,Enrichr_Submissions_TF-Gene_Coocurrence,ZNF488,1.481598e-07,0.000120,166.950324,10,"[TGFB2, TINAGL1, KLF5, IGFBP5, STC2, PEG10, FANCA, SLC2A3, DKK1, A..."
3,Rummagene_kinases,PMC5302127-NIHMS832530-supplement-supp table4.xlsx-CSNK1A1-ANKRD52...,1.828872e-07,0.000129,85.411577,17,"[COL17A1, TGFB2, TINAGL1, PARM1, GLCE, SLC2A3, DKK1, PRSS12, EFR3B..."
4,SubCell_BarCode,H322 Cytosol C2 30609389,5.509667e-06,0.000391,63.940988,13,"[GEN1, IGFBP5, STARD7, KLHL26, IFFO2, VWDE, SELENBP1, AMMECR1L, JM..."
5,Enrichr_Submissions_TF-Gene_Coocurrence,ZNF804A,1.560721e-06,0.000507,125.350876,9,"[TGFB2, PARM1, IGFBP5, PEG10, CHN1, ANK2, SLC2A3, DKK1, PCDH17]"
6,Enrichr_Submissions_TF-Gene_Coocurrence,ZNF667,1.560721e-06,0.000507,125.350876,9,"[TGFB2, TINAGL1, IGFBP5, STC2, JMY, FANCA, ANK2, SLC2A3, MELTF]"
7,Enrichr_Submissions_TF-Gene_Coocurrence,DZIP1,1.560721e-06,0.000507,125.350876,9,"[TGFB2, TINAGL1, GNG2, LRP1, IGFBP5, PEG10, FANCA, ANK2, SLC2A3]"
8,Enrichr_Submissions_TF-Gene_Coocurrence,ETV1,1.455874e-05,0.000816,91.088727,8,"[TGFB2, TINAGL1, GNG2, IGFBP5, PEG10, CHN1, FANCA, ANK2]"
9,Enrichr_Submissions_TF-Gene_Coocurrence,ELF5,1.455874e-05,0.000816,91.088727,8,"[SELENBP1, TGFB2, TINAGL1, KLF5, IGFBP5, FANCA, AGR2, MELTF]"


In [10]:
for d in DIRECTIONS:
    print(f"\n========== R239G {d} ==========")
    display(top_terms("R239G", d))


========== R239G up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,HumanCyc_2015,lysine degradation II (pipecolate pathway),0.000107,0.000640,1753.856190,2,"[AADAT, DHTKD1]"
1,HumanCyc_2016,lysine degradation II (pipecolate pathway) Homo sapiens PWY66-425,0.000107,0.000853,1753.856190,2,"[AADAT, DHTKD1]"
2,HumanCyc_2015,L-kynurenine degradation,0.000548,0.001643,523.470491,2,"[AADAT, DHTKD1]"
3,PheWeb_2019,Fracture of unspecified part of femur,0.000017,0.002133,802.636780,3,"[SMAD2, EPHA7, HMBOX1]"
4,HumanCyc_2016,L-kynurenine degradation Homo sapiens PWY-6309,0.000548,0.002190,523.470491,2,"[AADAT, DHTKD1]"
5,Proteomics_Drug_Atlas_2023,VU 0364439 Up,0.000002,0.002607,231.886685,6,"[SOCS2, BCL9, PPP1R15B, ATP7B, TRAF3IP2, PDE8A]"
6,Azimuth_2023,Kidney-L1-Schwann,0.000254,0.003563,906.709784,2,"[STARD13, ZNF536]"
7,ChEA_2022,OLIG2 23332759 ChIP-Seq OLIGODENDROCYTES Mouse,0.000006,0.003709,56.051584,16,"[EPHA5, NCOA1, SMAD2, PPP1R15B, EPHA7, STARD13, CCNH, PCDH20, URM1..."
8,GO_Biological_Process_2026,Regulation of Cellular Component Organization (GO:0051128),0.000006,0.004279,182.043066,6,"[SOCS2, TRAF3IP2, SLITRK4, WNT5A, DAB2IP, GFAP]"
9,NCI-Nature_2016,EphrinA-EPHA pathway Homo sapiens 1fd12dbd-6192-11e5-8ac5-06603eb7...,0.000198,0.004362,1089.782995,2,"[EPHA5, EPHA7]"



========== R239G down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,PerturbAtlas_MouseGenePerturbationSigs,PERTURB 1439 KO CCNG2 SRP059378 UP,7.783518e-11,2.788835e-07,3044.036638,6,"[ITIH2, AHSG, SERPINF2, APOA1, AFP, LCP1]"
1,RummaGEO_DrugPerturbations_2025,"GSE202434,GSE202438 9 V 5 TALAZOPARIB HUMAN DN",1.801581e-10,7.863901e-07,115.355026,29,"[COL17A1, CSTB, TINAGL1, AMIGO2, TULP3, PPP1R9A, HDAC9, CYTH2, FUT..."
2,Enrichr_Submissions_TF-Gene_Coocurrence,ZNF365,8.507953e-09,3.460610e-06,198.013499,12,"[CSTB, GAP43, NNMT, TINAGL1, IGFBP5, AMIGO2, TXNIP, PMEPA1, NRG1, ..."
3,Enrichr_Submissions_TF-Gene_Coocurrence,BNC1,8.507953e-09,3.460610e-06,198.013499,12,"[CSTB, CEMIP, GPRC5A, NNMT, TINAGL1, IGFBP5, AMIGO2, TXNIP, NRG1, ..."
4,Enrichr_Submissions_TF-Gene_Coocurrence,TFCP2L1,8.507953e-09,3.460610e-06,198.013499,12,"[CSTB, GPRC5A, TINAGL1, IGFBP5, TXNIP, PMEPA1, SORBS2, SLC2A3, SFN..."
5,Enrichr_Submissions_TF-Gene_Coocurrence,ZBED2,8.507953e-09,3.460610e-06,198.013499,12,"[COL17A1, CSTB, CEMIP, GPRC5A, TINAGL1, AMIGO2, IL18, TXNIP, PMEPA..."
6,GO_Molecular_Function_2021,peptidase inhibitor activity (GO:0030414),2.224386e-08,3.803701e-06,744.698624,6,"[CSTB, ITIH2, AHSG, PZP, SERPINF2, A2M]"
7,GO_Molecular_Function_2021,endopeptidase regulator activity (GO:0061135),5.313527e-08,4.543066e-06,601.531975,6,"[CSTB, ITIH2, AHSG, PZP, SERPINF2, A2M]"
8,NIBR_DRUGseq_2025_down,CID136263539 LE-01-MT37 BAID8 10UM,1.756162e-09,5.259704e-06,347.870130,10,"[CYTH2, GPRC5A, NNMT, TINAGL1, IGFBP5, AMIGO2, IL18, CXCL14, SAV1,..."
9,GO_Molecular_Function_2026,Peptidase Inhibitor Activity (GO:0030414),3.495455e-08,6.431638e-06,666.664321,6,"[CSTB, ITIH2, AHSG, PZP, SERPINF2, A2M]"
